In this notebook, we initialize a pretrained model re-trained on the 'bbox' dataset and then fine-tune it on the retinotopic images from the 'focus' dataset. We then evaluate its performance on the validation set and visualize the training dynamics across epochs.

In [1]:
import retinoto_py as fovea
model_name = 'convnext_base'
args = fovea.Params(do_fovea=True, model_name=model_name)
print(args)

Params(batch_size=128, num_workers=0, prefetch_factor=0, image_size=224, grid_size_ecc=161, grid_size_ang=309, do_mask=False, do_fovea=True, use_hexagonal_grid=True, rs_min=-3.5, rs_max=0.7, angle_start=-5.235987755982989, angle_margin=0.010166966516471823, mode='bilinear', padding_mode='zeros', model_name='convnext_base', num_epochs=20, subset_factor=1, optimizer_name='adamw', loss_name='CrossEntropyLoss', base_lr=1e-06, final_lr=1e-09, num_warmup_epochs=20, delta1=0.3, delta2=0.002, weight_decay=0.0001, label_smoothing=0.0005, do_full_training=True, do_augment=True, augment_proba=0.85, stochastic_depth_prob=0.6, seed=1998, shuffle=True, verbose=False)


# learning the `convnext_base` on the 'focus' retinotopic images

In [2]:
%ls -lh cached_data/45_*

-rw-r-----@ 1 laurent  staff   3,8K  7 août  18:04 cached_data/45_fovea_model_name=convnext_base_dataset=focus.json
-rw-r-----@ 1 laurent  staff   1,0G  7 août  18:04 cached_data/45_fovea_model_name=convnext_base_dataset=focus.pth


In [3]:
# %rm cached_data/45_focus_model_name*  # FORCING RECOMPUTE
# %rm cached_data/45_focus_model_name*.lock  # FORCING RECOMPUTE

In [4]:
dataset = 'bbox'
init_model_filename = args.data_cache / f'32_fovea_model_name={args.model_name}_dataset={dataset}.pth'

In [5]:
dataset = 'focus'
name = f'45_fovea_model_name={args.model_name}_dataset={dataset}'
model_filename, json_filename = fovea.do_learning(args, dataset, name, init_model_filename=init_model_filename)

Load JSON from pre-trained resnet cached_data/45_fovea_model_name=convnext_base_dataset=focus.json
cached_data/45_fovea_model_name=convnext_base_dataset=focus.pth: latest accuracy = 0.901


FileNotFoundError: [Errno 2] No such file or directory: '/Users/laurent/data/Imagenet/Imagenet_focus/train'

# Evaluation on the validation dataset

In [ ]:
results = fovea.pd.read_json(args.data_cache / f'45_fovea_model_name={model_name}_dataset={dataset}.json')
print(model_name, dataset, '- Accuracy=', results.tail(1)['acc_val'].item())

In [ ]:
model_name, dataset, len(results)

In [ ]:
results.tail(10)

## Plot learning evolution

In [ ]:
fig, ax = fovea.plt.subplots()
color = 'r'
lw = 1

json_filename = args.data_cache / f'45_fovea_model_name={model_name}_dataset={dataset}.json'
# model_filename, json_filename = fovea.do_learning(args, dataset, name)

df_train = fovea.pd.read_json(json_filename, orient='records')

ax = df_train.plot(x='epoch', y='acc_train', 
                    c=color, ls='dashed', lw=lw,
                    grid=True, ax=ax, label='TRAIN: ' + model_name)    
ax = df_train.plot(x='epoch', y='acc_val', 
                    c=color, lw=lw,
                    grid=True, ax=ax, label='VAL: ' + model_name)   